### Agentes LLM: herramientas, memoria, RAG y multiagentes


El propósito de este cuaderno es construir una visión aplicada de cómo un sistema basado en LLM puede pasar de responder texto a **tomar decisiones operativas** mediante herramientas, memoria, planificación, verificación y recuperación de evidencia.

#### Ruta de aprendizaje
- herramientas y contratos de entrada/salida
- memoria de trabajo y traza de ejecución
- planificación y descomposición de tareas
- verificación, guardrails y evaluación básica
- ReAct como ciclo de control
- conexión con RAG y agentic RAG
- introducción a sistemas multiagente
- plantilla opcional con `smolagents`.

#### Conceptos técnicos clave
- **policy (política)**: componente que decide la siguiente acción.
- **state (estado)**: información acumulada durante la ejecución.
- **tool use (uso de herramientas)**: invocación controlada de funciones externas.
- **grounding**: anclar una respuesta en cálculo, evidencia o reglas.
- **orchestration (orquestación)**: coordinación de pasos, herramientas o agentes.

### 1. Mapa de la semana 10

##### Bloque A. Agentes
- herramientas
- memoria
- planificación
- verificación
- guardrails
- ReAct

##### Bloque B. Retrieval y RAG
- embeddings
- semantic search
- vector stores
- chunking
- top-k retrieval
- grounded generation
- agentic RAG como extensión

#### Bloque C. Multiagentes
- especialización de roles
- coordinación mediante supervisor
- memoria compartida
- crítica y validación cruzada
- handoffs entre agentes

##### Idea central

Un **LLM puro** genera texto.

Un **agente con LLM**:
1. decide qué hacer,
2. usa herramientas si hace falta,
3. observa el resultado,
4. actualiza memoria,
5. verifica antes de cerrar.

##### Conceptos técnicos clave
- **control loop (bucle de control)**: ciclo decisión -> acción -> observación -> actualización.
- **tool schema (esquema de herramienta)**: contrato formal de argumentos y salida de una herramienta.
- **planner (planificador)**: módulo que descompone una tarea en subtareas.
- **evaluator/critic (evaluador/crítico)**: módulo que revisa calidad, consistencia o seguridad.

#### 2. ¿Qué es un agente?

##### Definición operativa

Un agente es un sistema donde el modelo no solo responde, sino que puede:

- elegir acciones,
- usar herramientas,
- leer observaciones,
- mantener memoria,
- seguir un plan,
- verificar antes de responder.

##### Patrón general

**Thought -> Action -> Observation -> Update Memory -> Next Step**

En este cuaderno, `Thought` se usa como **etiqueta didáctica de la decisión operativa**, no como exposición de razonamiento interno real de un modelo.

##### Conceptos técnicos clave
- **acción**: operación seleccionada por la política del agente.
- **observación**: resultado devuelto por una herramienta o por el entorno.
- **entorno**: fuente externa con la que el agente interactúa.
- **estado del agente**: memoria acumulada de tarea, pasos y resultados.

#### 3. Entorno de trabajo

##### En este cuaderno

Usaremos tres niveles:

1. **Nivel local y autocontenido**  
   Un mini agente que funciona sin depender de un modelo remoto.

2. **Nivel multiagente simulado**  
   Un sistema con roles especializados que colaboran sobre una misma tarea.

3. **Nivel framework**  
   Una plantilla opcional con `smolagents`, útil si luego quieres pasar a un agente más realista.

##### Conceptos técnicos clave
- **mock agent (agente simulado)**: agente simulado para estudiar arquitectura sin coste de API.
- **deterministic routing (Enrutamiento determinista)**: selección de acciones mediante reglas reproducibles.
- **framework agent (agente de framework)**: agente implementado con librerías que gestionan herramientas, memoria y llamadas al modelo.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
import re

#### 4. Herramientas mínimas


Una herramienta convierte una petición ambigua en una operación verificable. En agentes LLM, las herramientas reducen la incertidumbre porque fuerzan una salida estructurada: cálculo, búsqueda, validación o consulta a una API.

Aquí definiremos tres:

- `calculator_tool`
- `kb_search_tool`
- `policy_check_tool`

##### Conceptos técnicos clave

- **tool calling (llamada a funciones)**: mecanismo por el cual el agente invoca una función externa.
- **input validation (validación de entrada)**: control de entradas antes de ejecutar una herramienta.
- **sandboxing (entorno controlado)**: limitar lo que una herramienta puede ejecutar.
- **determinismo**: una misma entrada produce una misma salida.

In [ ]:
def calculator_tool(expression: str) -> str:
    """Evalúa expresiones aritméticas simples de forma controlada."""
    allowed = re.fullmatch(r"[0-9\s\+\-\*\/\(\)\.]+", expression)
    if not allowed:
        return "ERROR: expresión no permitida"
    try:
        value = eval(expression, {"__builtins__": {}}, {})
        return f"RESULTADO: {value}"
    except Exception as e:
        return f"ERROR: {e}"

KNOWLEDGE_BASE = {
    "rag": "RAG combina recuperación de evidencia externa con generación del LLM.",
    "embedding": "Un embedding es una representación vectorial densa de un texto.",
    "react": "ReAct estructura el ciclo pensamiento, acción y observación.",
    "guardrail": "Un guardrail restringe entradas, salidas, permisos o acciones del sistema.",
    "vector store": "Un vector store guarda embeddings junto con texto y metadatos.",
    "multiagente": "Un sistema multiagente coordina varios agentes especializados para resolver una tarea.",
    "orquestador": "Un orquestador coordina agentes, herramientas, memoria compartida y criterios de parada.",
    "critic": "Un agente critic revisa consistencia, evidencia, riesgos y calidad de una respuesta.",
    "blackboard": "Una memoria blackboard permite que varios agentes lean y escriban hallazgos compartidos.",
}

def kb_search_tool(query: str) -> str:
    """Búsqueda simple sobre una base pequeña por coincidencia de palabras clave."""
    q = query.lower()
    hits = []
    for key, value in KNOWLEDGE_BASE.items():
        if key in q or any(tok in value.lower() for tok in q.split()):
            hits.append((key, value))
    if not hits:
        return "SIN_EVIDENCIA"
    return "\n".join([f"[{k}] {v}" for k, v in hits[:3]])

def policy_check_tool(answer: str) -> str:
    """Validador simple de salida."""
    banned_patterns = ["contraseña", "password", "api_key", "token secreto"]
    lowered = answer.lower()
    for pat in banned_patterns:
        if pat in lowered:
            return f"RECHAZADO: contiene patrón sensible -> {pat}"
    if len(answer.strip()) == 0:
        return "RECHAZADO: salida vacía"
    return "APROBADO"

#### 5. Memoria del agente

##### Qué queremos guardar

- tarea original,
- pasos ejecutados,
- acciones,
- observaciones,
- respuesta final.

##### Tipos de memoria

- **memoria de trabajo**: información temporal de la tarea actual.
- **memoria episódica**: historial de interacciones o trazas anteriores.
- **memoria semántica**: conocimiento persistente, normalmente indexado.
- **memoria externa**: base de datos, vector store, archivos o logs.

##### Conceptos técnicos clave
- **trace (traza)**: registro secuencial de pasos del agente.
- **stateful execution (ejecución dependiente del contexto acumulado)**: ejecución que depende de un estado acumulado.
- **auditability (auditablidad)**: posibilidad de revisar qué hizo el sistema y por qué.

In [ ]:
@dataclass
class Step:
    thought: str
    action: str
    observation: str

@dataclass
class AgentMemory:
    task: str
    steps: List[Step] = field(default_factory=list)
    final_answer: Optional[str] = None

    def add(self, thought: str, action: str, observation: str):
        self.steps.append(Step(thought=thought, action=action, observation=observation))

    def show(self):
        print(f"TAREA: {self.task}\n")
        for i, step in enumerate(self.steps, start=1):
            print(f"Paso {i}")
            print(f"  Thought     : {step.thought}")
            print(f"  Action      : {step.action}")
            print(f"  Observation : {step.observation}")
            print()
        if self.final_answer is not None:
            print("RESPUESTA FINAL:")
            print(self.final_answer)

#### 6. Un agente simple

Construiremos un agente pequeño, sin usar un LLM remoto, pero con la lógica básica de un agente real.

##### Arquitectura usada

- `choose_action()` funciona como política simple.
- `calculator_tool()` resuelve operaciones numéricas.
- `kb_search_tool()` recupera evidencia textual.
- `policy_check_tool()` valida la salida antes de entregarla.
- `AgentMemory` guarda la traza.

##### Conceptos técnicos clave
- **router (enrutador)**: función que selecciona qué herramienta usar.
- **synthesis (síntesis)**: generación de respuesta final usando observaciones.
- **post-condition check (verificación post-condición)**: verificación al final de la ejecución.

In [ ]:
def choose_action(task: str, memory: AgentMemory) -> str:
    text = task.lower()

    if re.search(r"[0-9]\s*[\+\-\*/]\s*[0-9]", text):
        exprs = re.findall(r"[0-9\s\+\-\*\/\(\)\.]+", task)
        expr = max(exprs, key=len).strip() if exprs else ""
        if expr:
            return f"calculator:{expr}"

    for kw in ["rag", "embedding", "react", "guardrail", "vector store", "multiagente", "orquestador", "critic", "blackboard"]:
        if kw in text:
            return f"kb_search:{kw}"

    return "respond:No necesito herramientas para esta tarea."

def synthesize_answer(task: str, memory: AgentMemory) -> str:
    observations = [s.observation for s in memory.steps]
    joined = "\n".join(observations)

    if "RESULTADO:" in joined:
        last_result = [obs for obs in observations if "RESULTADO:" in obs][-1]
        return f"Respuesta verificada con herramienta matemática. {last_result}"

    if "SIN_EVIDENCIA" not in joined and len(observations) > 0:
        return "Respuesta basada en evidencia recuperada:\n" + joined

    return "Respuesta directa: no encontré evidencia externa ni cálculo requerido."

def simple_agent(task: str, max_steps: int = 3, verbose: bool = True) -> AgentMemory:
    memory = AgentMemory(task=task)

    for _ in range(max_steps):
        action = choose_action(task, memory)

        if action.startswith("calculator:"):
            expr = action.split(":", 1)[1]
            observation = calculator_tool(expr)
            memory.add("Necesito verificar el cálculo con una herramienta.", action, observation)
            break

        if action.startswith("kb_search:"):
            query = action.split(":", 1)[1]
            observation = kb_search_tool(query)
            memory.add("Necesito recuperar conocimiento antes de responder.", action, observation)
            break

        if action.startswith("respond:"):
            observation = action.split(":", 1)[1]
            memory.add("La tarea parece resolverse sin herramientas.", action, observation)
            break

    draft = synthesize_answer(task, memory)
    policy_result = policy_check_tool(draft)
    memory.add(
        "Antes de cerrar, verifico la salida con una política simple.",
        "policy_check",
        policy_result
    )

    memory.final_answer = draft if policy_result.startswith("APROBADO") else "La respuesta fue bloqueada por el validador."

    if verbose:
        memory.show()

    return memory

#### 7. Pruebas rápidas del agente

Se debe validar tres rutas de ejecución:

- una consulta matemática,
- una consulta conceptual,
- una respuesta directa sin herramientas.

##### Conceptos técnicos clave
- **smoke test (prueba de humo)**: prueba rápida para confirmar que el flujo básico funciona.
- **test case (casos de tesis)**: entrada diseñada para activar una rama específica del agente.
- **expected behavior (comportamiento esperado)**: comportamiento esperado para una clase de tareas.

In [ ]:
memory_math = simple_agent("¿Cuánto es 47 / 12 * 3.14?")

In [ ]:
memory_concept = simple_agent("Explica qué es RAG y para qué sirve.")

In [ ]:
memory_direct = simple_agent("Salúdame en una línea.")

#### 8. Planificación simple

##### De tarea única a subtareas

La planificación permite convertir una petición general en pasos ejecutables. En sistemas reales, el plan puede ser generado por un LLM, por reglas, por un workflow fijo o por una combinación de ambos.

##### Conceptos técnicos clave
- **task decomposition (descomposición de tareas)**: dividir una tarea compleja en subtareas.
- **plan validation (validación de plan)**: revisar que el plan sea ejecutable.
- **sequential planning (planificación secuencial)**: pasos ordenados linealmente.
- **dynamic replanning (replanificación dinámica)**: modificar el plan cuando una observación cambia el contexto.

In [ ]:
def plan_task(task: str) -> List[str]:
    plan = ["interpretar_pedido"]

    if re.search(r"[0-9]\s*[\+\-\*/]\s*[0-9]", task.lower()):
        plan.append("usar_calculadora")
    elif any(k in task.lower() for k in ["rag", "embedding", "react", "guardrail", "vector store", "multiagente", "orquestador", "critic", "blackboard"]):
        plan.append("recuperar_evidencia")
    else:
        plan.append("responder_directo")

    plan.append("verificar_salida")
    return plan

plan_task("Explica qué es un vector store.")

#### 9. ReAct en versión corta

##### Patrón
- Thought
- Action
- Observation

ReAct combina razonamiento operativo y uso de herramientas en un ciclo repetible. En la práctica, ayuda a separar la decisión de la evidencia observada.

##### Conceptos técnicos clave
- **reason-act loop (ciclo razonar-actuar)**: ciclo alternado de decisión y acción.
- **observation grounding (anclaje por observación)**: usar observaciones para reducir respuestas inventadas.
- **intermediate step (paso intermedio)**: paso interno antes de la respuesta final.

In [ ]:
def react_trace(task: str):
    memory = AgentMemory(task=task)

    if "react" in task.lower():
        memory.add(
            "La consulta pide definición técnica. Buscaré evidencia.",
            "kb_search:react",
            kb_search_tool("react")
        )
    else:
        memory.add(
            "Primero intento recuperar evidencia relevante.",
            "kb_search:rag",
            kb_search_tool("rag")
        )

    draft = synthesize_answer(task, memory)
    memory.add(
        "Con la observación disponible, produzco una respuesta final.",
        "respond",
        draft
    )
    memory.final_answer = draft
    memory.show()

react_trace("¿Qué es ReAct?")

#### 10. Verificación y guardrails

##### Dos ideas distintas

**Verificación**
- ¿la respuesta coincide con cálculo, evidencia o reglas?

**Guardrails**
- ¿la respuesta respeta restricciones de seguridad, permisos y formato?

##### Capas típicas de control
- validación de entrada,
- permisos de herramienta,
- filtros de salida,
- revisión con evidencia,
- límites de longitud, coste o tiempo.

##### Conceptos técnicos clave
- **policy enforcement (aplicación de políticas )**: aplicación de reglas obligatorias.
- **evidence matching (correspondencia de evidencia)**: contraste entre respuesta y fuente.
- **output constraint (restricción de salida)**: restricción sobre formato, contenido o longitud.

In [ ]:
def verify_with_evidence(answer: str, evidence: str) -> bool:
    evidence_terms = set(re.findall(r"[a-zA-ZáéíóúÁÉÍÓÚ]+", evidence.lower()))
    answer_terms = set(re.findall(r"[a-zA-ZáéíóúÁÉÍÓÚ]+", answer.lower()))
    return len(evidence_terms & answer_terms) >= 3

evidence = kb_search_tool("rag")
answer = "RAG combina recuperación de evidencia externa con generación del LLM."
verify_with_evidence(answer, evidence)

#### 11. Conexión con RAG

#### RAG como herramienta del agente

RAG no compite con agentes. Muchas veces **RAG es una herramienta dentro de un agente**.

#### Pipeline típico de RAG
1. dividir documentos en chunks,
2. generar embeddings,
3. indexar en un vector store,
4. recuperar los `top-k` chunks relevantes,
5. generar una respuesta citando o usando la evidencia.

##### Agentic RAG
En **agentic RAG**, el agente puede decidir cuándo buscar, reformular la consulta, recuperar más evidencia, usar otra herramienta o pedir verificación.

#####  Conceptos técnicos clave
- **chunking (fragmentación)**: división de documentos en unidades recuperables.
- **embedding**: representación vectorial de texto.
- **retriever (recuperador)**: componente que selecciona evidencia.
- **reranking**: reordenamiento de resultados recuperados.
- **grounded generation (generación anclada)**: generación condicionada por evidencia.

In [ ]:
def rag_tool(query: str) -> Dict[str, Any]:
    docs = []
    q = query.lower()
    for key, value in KNOWLEDGE_BASE.items():
        if key in q or any(tok in value.lower() for tok in q.split()):
            docs.append({"key": key, "text": value})
    return {"query": query, "top_k": len(docs[:3]), "documents": docs[:3]}

rag_tool("vector store y embedding")

#### 12. Introducción a sistemas multiagente

Un sistema multiagente divide una tarea entre varios agentes especializados. En lugar de tener un solo agente que decide, busca, calcula y valida, se separan responsabilidades.

##### Roles frecuentes
- **PlannerAgent**: descompone la tarea en subtareas.
- **ResearchAgent**: recupera evidencia o contexto.
- **ExecutorAgent**: ejecuta herramientas concretas.
- **CriticAgent**: revisa errores, riesgos y consistencia.
- **Orchestrator**: coordina el orden de ejecución y el criterio de parada.

##### Patrones de coordinación
- **supervisor-worker**: un supervisor asigna tareas a agentes trabajadores.
- **pipeline**: cada agente transforma la salida del agente anterior.
- **debate/critique**: varios agentes comparan respuestas y un crítico selecciona o mejora.
- **blackboard**: todos leen y escriben en una memoria compartida.

##### Conceptos técnicos clave
- **handoff**: transferencia de una subtarea entre agentes.
- **shared memory (memoria compartida)**: memoria común usada por varios roles.
- **consensus (consenso)**: mecanismo para acordar una salida final.
- **coordination overhead (sobrecarga de coordinación)**: coste adicional de coordinar múltiples agentes.

In [ ]:
@dataclass
class MultiAgentState:
    task: str
    blackboard: Dict[str, Any] = field(default_factory=dict)
    log: List[str] = field(default_factory=list)

    def write(self, key: str, value: Any, author: str):
        self.blackboard[key] = value
        self.log.append(f"{author} -> {key}: {value}")

def planner_agent(state: MultiAgentState) -> MultiAgentState:
    """Divide la tarea en subtareas ejecutables."""
    plan = ["interpretar_tarea"]

    text = state.task.lower()
    if any(k in text for k in ["rag", "embedding", "react", "guardrail", "vector store", "multiagente"]):
        plan.append("recuperar_evidencia")
    if re.search(r"[0-9]\s*[\+\-\*/]\s*[0-9]", text):
        plan.append("calcular")
    plan.append("criticar_respuesta")
    plan.append("sintetizar_respuesta")

    state.write("plan", plan, "PlannerAgent")
    return state

def research_agent(state: MultiAgentState) -> MultiAgentState:
    """Busca evidencia conceptual en la base de conocimiento."""
    text = state.task.lower()
    candidates = ["multiagente", "orquestador", "critic", "blackboard", "rag", "embedding", "react", "guardrail", "vector store"]

    evidence = []
    for keyword in candidates:
        if keyword in text:
            evidence.append(kb_search_tool(keyword))

    if not evidence:
        evidence.append("SIN_EVIDENCIA")

    state.write("evidence", "\n".join(evidence), "ResearchAgent")
    return state

def executor_agent(state: MultiAgentState) -> MultiAgentState:
    """Ejecuta herramientas deterministas cuando la tarea lo requiere."""
    exprs = re.findall(r"[0-9\s\+\-\*\/\(\)\.]+", state.task)
    expr = max(exprs, key=len).strip() if exprs else ""

    if expr and re.search(r"[0-9]\s*[\+\-\*/]\s*[0-9]", expr):
        result = calculator_tool(expr)
    else:
        result = "NO_CALCULATION_REQUIRED"

    state.write("calculation", result, "ExecutorAgent")
    return state

def critic_agent(state: MultiAgentState) -> MultiAgentState:
    """Evalúa si hay evidencia suficiente y si el cálculo fue seguro."""
    evidence = state.blackboard.get("evidence", "")
    calculation = state.blackboard.get("calculation", "")

    checks = {
        "has_evidence": "SIN_EVIDENCIA" not in evidence,
        "calculation_ok": not calculation.startswith("ERROR"),
        "has_plan": "plan" in state.blackboard,
    }
    state.write("critique", checks, "CriticAgent")
    return state

def synthesizer_agent(state: MultiAgentState) -> MultiAgentState:
    """Integra plan, evidencia, cálculo y crítica en una respuesta breve."""
    evidence = state.blackboard.get("evidence", "")
    calculation = state.blackboard.get("calculation", "")
    critique = state.blackboard.get("critique", {})

    answer_parts = [
        "Respuesta multiagente:",
        f"- Evidencia: {evidence}",
        f"- Cálculo: {calculation}",
        f"- Revisión: {critique}",
    ]
    final_answer = "\n".join(answer_parts)
    policy_result = policy_check_tool(final_answer)

    state.write("final_answer", final_answer if policy_result == "APROBADO" else "Salida bloqueada por política.", "SynthesizerAgent")
    return state

def run_multiagent_system(task: str) -> MultiAgentState:
    """Orquestador secuencial tipo pipeline."""
    state = MultiAgentState(task=task)

    for agent in [planner_agent, research_agent, executor_agent, critic_agent, synthesizer_agent]:
        state = agent(state)

    return state

multi_state = run_multiagent_system("Explica qué es un sistema multiagente y calcula 47 / 12 * 3.14")
multi_state.blackboard["final_answer"]

#### 13. Lectura de la traza multiagente

##### Qué observar

La traza permite ver qué escribió cada agente en la memoria compartida. Esto es útil para depuración, evaluación y sustentación oral.

##### Conceptos técnicos clave
- **blackboard inspection**: revisión de la memoria compartida.
- **role attribution (atribución de rol )**: identificar qué agente produjo cada dato.
- **debug trace (traza de depuración)**: secuencia de eventos usada para encontrar errores.

In [ ]:
for event in multi_state.log:
    print(event)

#### 14. Plantilla opcional con smolagents


La documentación de Hugging Face usa `smolagents` para construir agentes con herramientas. Para herramientas simples, la documentación recomienda el decorador `@tool`. Cuando el flujo se vuelve más complejo, los frameworks ayudan con herramientas, parser de llamadas, memoria y manejo de errores.

##### Conceptos técnicos clave
- **agent framework (framework de agente)**: librería que abstrae llamadas al modelo, herramientas y memoria.
- **tool decorator (decorador de herramientas)**: forma de registrar una función como herramienta usable por el agente.
- **parser de acciones**: componente que interpreta qué herramienta pidió usar el modelo.
- **runtime**: entorno donde se ejecutan pasos, herramientas y validaciones.

In [ ]:
SMOLAGENTS_TEMPLATE = r'''# %pip install -q smolagents huggingface_hub

from smolagents import CodeAgent, InferenceClientModel, tool

@tool
def calculator(expression: str) -> str:
    """
    Evalúa una expresión aritmética simple.
    Args:
        expression: Expresión matemática con +, -, *, / y paréntesis.
    """
    import re
    allowed = re.fullmatch(r"[0-9\s\+\-\*\/\(\)\.]+", expression)
    if not allowed:
        return "ERROR: expresión no permitida"
    return str(eval(expression, {"__builtins__": {}}, {}))

@tool
def mini_kb(query: str) -> str:
    """
    Busca definiciones cortas sobre agentes y RAG.
    Args:
        query: Consulta textual.
    """
    kb = {
        "rag": "RAG combina retrieval y generación.",
        "react": "ReAct usa thought, action y observation.",
        "guardrail": "Un guardrail restringe acciones o salidas."
    }
    q = query.lower()
    for k, v in kb.items():
        if k in q:
            return v
    return "SIN_EVIDENCIA"

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    temperature=0.2,
    max_tokens=1024,
)

agent = CodeAgent(
    model=model,
    tools=[calculator, mini_kb],
    max_steps=5,
    verbosity_level=1,
)

result = agent.run("Explica qué es RAG y calcula 47/12*3.14")
print(result)
'''
print(SMOLAGENTS_TEMPLATE)

#### 15. Memoria en un framework


En frameworks de agentes, la memoria suele registrar:
- tarea
- pasos
- observaciones
- errores
- respuesta final

##### Conceptos técnicos clave
- **conversation state (estado de conversación)**: estado de la conversación o tarea.
- **execution log (registro de ejecución)**: historial estructurado de acciones.
- **checkpointing (punto de control)**: guardar puntos intermedios para depurar o reanudar.
- **long-term memory (memoria de largo plazo)**: memoria persistente fuera del contexto inmediato.

#### 16. Ejercicios

##### Básicos
1. Ejecuta el agente con una consulta matemática distinta y revisa la traza.
2. Agrega una nueva entrada a `KNOWLEDGE_BASE` y prueba una nueva consulta.
3. Modifica `policy_check_tool` para bloquear respuestas demasiado largas.

##### Intermedios
4. Haz que el agente use primero `kb_search_tool` y luego `calculator_tool` si la tarea pide definición y cálculo.
5. Agrega una herramienta `unit_converter_tool`.
6. Modifica `plan_task()` para devolver subtareas más detalladas.

##### Multiagentes
7. Agrega un agente `SecurityAgent` que revise si una respuesta viola una política.
8. Cambia el sistema multiagente para que el `PlannerAgent` devuelva más de tres subtareas.
9. Implementa una memoria compartida tipo `blackboard` con claves `plan`, `evidence`, `calculation` y `critique`.

##### De análisis
10. Explica la diferencia entre LLM base, agente, RAG y agentic RAG.
11. Da un ejemplo donde RAG simple sea suficiente.
12. Da un ejemplo donde sí necesites un agente con herramientas.
13. Compara un workflow fijo con un sistema multiagente.

##### Sustentación
14. ¿Por qué una herramienta puede reducir alucinaciones?
15. ¿Qué diferencia hay entre memoria y contexto?
16. ¿Qué problema resuelven los guardrails?
17. ¿Por qué un agente no siempre es mejor que un workflow fijo?
18. ¿Cuándo conviene usar RAG como herramienta del agente?
19. ¿Qué riesgos aparecen cuando varios agentes comparten memoria?

##### Tus respuestas